# Run Models

In [1]:
# Import Libraries
%run /Users/u4eeevmq/Documents/Python/HyporheicFloPy/VQuintana/notebooks/common_imports.py

# Retrieve stored variables
%store -r md6_exe_path
%store -r md7_exe_path
%store -r sim_name
%store -r workspace
%store -r figs_path
%store -r gwf_name
%store -r mp7_name
%store -r gwf_ws
%store -r mp7_ws
%store -r headfile
%store -r head_filerecord
%store -r budgetfile
%store -r budget_filerecord
%store -r write
%store -r run
%store -r plot
%store -r plot_show
%store -r plot_save

# Retrieve model parameters
%store -r length_units
%store -r time_units
%store -r nper
%store -r cell_size_x
%store -r cell_size_y
%store -r gw_mod_depth
%store -r z
%store -r kh
%store -r kv
%store -r gw_offset
%store -r porosity
%store -r rch_iface
%store -r rch_iflowface
%store -r recharge_rate
%store -r nstp
%store -r perlen
%store -r tsmult

# Retrieve spatial data
%store -r hec_ras_crs
%store -r terrain_elevation
%store -r raster_transform
%store -r transform
%store -r raster_crs
%store -r terrain_output_raster
%store -r water_surface_output_raster
%store -r cropped_output_raster
%store -r ground_water_domain
%store -r left_boundary
%store -r right_boundary

# retrieve model domain data
%store -r terrain_elevation
%store -r raster_transform
%store -r raster_crs
%store -r raster_bounds_box
%store -r bed_elevation
%store -r raster_width
%store -r raster_height
%store -r ncol
%store -r nrow
%store -r top
%store -r nlay
%store -r grid_x
%store -r grid_y
%store -r grid_points
%store -r intersecting_points
%store -r xorigin
%store -r yorigin
%store -r xmin
%store -r ymin
%store -r xmax
%store -r ymax
%store -r tops
%store -r botm

# retrieve model boundary data
%store -r left_start
%store -r left_end
%store -r right_start
%store -r right_end
%store -r upstream_start_x
%store -r upstream_start_y
%store -r upstream_end_x
%store -r upstream_end_y
%store -r downstream_start_x
%store -r downstream_start_y
%store -r downstream_end_x
%store -r downstream_end_y
%store -r upstream_line
%store -r downstream_line
%store -r upstream_boundary
%store -r downstream_boundary
%store -r grid_cells
%store -r grid_gdf
%store -r idomain

# retrieve model boundary conditions
%store -r boundary_cells
%store -r left_boundary_cells
%store -r right_boundary_cells
%store -r upstream_boundary_cells
%store -r downstream_boundary_cells
%store -r all_boundary_cells
%store -r unique_boundary_cells
%store -r left_boundary_cells_first_layer
%store -r right_boundary_cells_first_layer
%store -r upstream_boundary_cells_first_layer
%store -r downstream_boundary_cells_first_layer
%store -r max_elevation_upstream
%store -r max_elevation_downstream
%store -r gw_elevation_left_first
%store -r gw_elevation_left_last
%store -r gw_elevation_right_first
%store -r gw_elevation_right_last
%store -r gw_elevation_upstream_first
%store -r gw_elevation_upstream_last
%store -r gw_elevation_downstream_first
%store -r gw_elevation_downstream_last
%store -r gw_elevation_left
%store -r gw_elevation_right
%store -r gw_elevation_upstream
%store -r gw_elevation_downstream
%store -r grid_points_coords
%store -r elevation_values
%store -r grid_points_df
%store -r output_csv
%store -r cropped_df
%store -r river_cells
%store -r river_x
%store -r river_y
%store -r river_elevation
%store -r chd_data
%store -r unique_chd_cells
%store -r duplicate_chd_cells
%store -r chd_data_converted

# retrieve well data
%store -r wel_data

# retrieve nodes
# %store -r nodes


c:\Users\u4eeevmq\Documents\Python\HyporheicFloPy\.venv\Lib\site-packages\geopandas\_compat.py:7: DeprecationWarning: The 'shapely.geos' module is deprecated, and will be removed in a future version. All attributes of 'shapely.geos' are available directly from the top-level 'shapely' namespace (since shapely 2.0.0).
  import shapely.geos


## MODFLOW 6 Groundwater Model

In [ ]:
#----------------------Model Setup Functions ------------------------#
def build_gwf_model(
    cfg: Any,                              # config/settings instance
    chd_data: Sequence[Sequence[float]],   # CHD rows/records
    idomain: np.ndarray,                   # (nlay, nrow, ncol) activity mask
) -> Tuple[flopy.mf6.MFSimulation, flopy.mf6.ModflowGwf]:
    """
    Construct a stand-alone MODFLOW 6 GWF model using information held
    in `cfg` plus the caller-supplied `chd_data` and `idomain` array.

    Returns
    -------
    (sim, gwf) tuple – ready for sim.write_simulation() and sim.run_simulation().
    """
    # --- Sanity checks -------------------------------------------------
    if cfg.nlay is None or cfg.nrow is None or cfg.ncol is None:
        raise ValueError("cfg.nlay / nrow / ncol must be populated first.")
    if idomain.shape != (cfg.nlay, cfg.nrow, cfg.ncol):
        raise ValueError("`idomain` dimensions don’t match cfg grid.")

    # --- 1. Simulation container ---------------------------------------
    sim = flopy.mf6.MFSimulation(
        sim_name = cfg.sim_name,
        exe_name = str(cfg.md6_exe_path),
        sim_ws   = str(cfg.gwf_ws),
    )

    flopy.mf6.ModflowTdis(
        sim,
        time_units = cfg.time_units.upper(),
        nper       = cfg.nper,
        perioddata = [(cfg.perlen, cfg.nstp, cfg.tsmult)],
    )

    # --- 2. Groundwater-flow model -------------------------------------
    gwf = flopy.mf6.ModflowGwf(
        sim,
        modelname  = cfg.gwf_name,
        save_flows = True,
    )

    flopy.mf6.ModflowGwfdis(
        gwf,
        nlay    = cfg.nlay,
        nrow    = cfg.nrow,
        ncol    = cfg.ncol,
        delr    = cfg.cell_size_x,
        delc    = cfg.cell_size_y,
        top     = cfg.tops[0],
        botm    = cfg.botm,
        idomain = idomain,
        xorigin = cfg.xmin,
        yorigin = cfg.ymin,
    )

    # Set or update the model grid crs and coordinate info
    gwf.modelgrid.crs = cfg.project_crs
    gwf.modelgrid.set_coord_info(cfg.xmin, cfg.ymin, crs=cfg.project_crs)

    # --- 3. Packages ---------------------------------------------------
    # 3-a. initial heads
    strt = np.full((cfg.nlay, cfg.nrow, cfg.ncol), cfg.bed_elevation, dtype=float)
    flopy.mf6.ModflowGwfic(gwf, strt=strt)

    # 3-b. hydraulic properties
    flopy.mf6.ModflowGwfnpf(
        gwf,
        icelltype = 2,
        k         = cfg.kh,
        k33       = cfg.kv,
        save_flows        = True,
        save_saturation   = True,
        save_specific_discharge = True,
    )

    # 3-c. constant-head boundaries
    if chd_data:
        flopy.mf6.ModflowGwfchd(
            gwf,
            maxbound           = len(chd_data),
            stress_period_data = {0: chd_data},
            save_flows         = True,
        )

    # 3-d. output control
    flopy.mf6.ModflowGwfoc(
        gwf,
        saverecord        = [("HEAD", "ALL"), ("BUDGET", "ALL")],
        head_filerecord   = [cfg.headfile],
        budget_filerecord = [cfg.budgetfile],
        printrecord       = [("HEAD", "LAST")],
    )

    # --- 4. IMS solver -------------------------------------------------
    flopy.mf6.ModflowIms(
        sim,
        print_option      = "SUMMARY",
        outer_dvclose     = 1e-4,
        outer_maximum     = 200,
        inner_maximum     = 500,
        inner_dvclose     = 1e-4,
        rcloserecord      = 1e-4,
        linear_acceleration = "BICGSTAB",
        relaxation_factor = 0.97,
    )

    # --- 5. Externalize big arrays for disk IO & QA --------------------
    external_dir = Path(gwf.model_ws) / "arrays"
    external_dir.mkdir(exist_ok=True)

    dis = gwf.get_package("DIS")
    ic  = gwf.get_package("IC")

    def _layered_records(array: np.ndarray, basename: str) -> list[dict]:
        return [{
            "filename": str(PurePath("arrays") / f"{basename}_L{lay+1}.bin"),
            "binary": True,
            "data": np.asarray(array[lay]),
            "iprn": 0,
            "factor": 1.0,
        } for lay in range(array.shape[0])]

    # Externalize DIS arrays
    if dis is not None:
        dis.top.set_record({
            "filename": str(PurePath("arrays") / "top.bin"),
            "binary": True,
            "data": np.asarray(dis.top.array),
            "iprn": 0,
            "factor": 1.0,
        })
        dis.botm.set_record(_layered_records(dis.botm.array, "botm"))
        if hasattr(dis, "idomain") and dis.idomain.array is not None:
            dis.idomain.set_record(_layered_records(dis.idomain.array, "idomain"))

    # Externalize IC array
    if ic is not None:
        ic.strt.set_record(_layered_records(ic.strt.array, "strt"))

    return sim, gwf

## MODPATH 7 Particle Tracking

In [ ]:
def build_particle_models(
    sim_name: str,
    gwf: flopy.mf6.ModflowGwf,
    river_cells: list[tuple[int, int, int]],
    *,
    mp7_ws: Path | str | None = None,
    exe_path: str | Path | None = None,
) -> tuple[Modpath7, Modpath7]:
    """
    Create **forward** *and* **backward** MODPATH 7 models that start particles
    in every cell listed in *river_cells*.

    Parameters
    ----------
    sim_name      : str
        Base name – “_mp_forward / _mp_backward” are appended automatically.
    gwf           : flopy.mf6.ModflowGwf
        The built groundwater-flow model whose grid MODPATH inherits.
    river_cells   : list[(k, i, j)]
        Sequence of (layer,row,col) indices where particles should be released.
    mp7_ws, exe_path
        Optional overrides.  If omitted workspace & exe are taken from *gwf*
        (``gwf.simulation.sim_path`` and ``os.getenv("MP7")``).

    Returns
    -------
    (mp_forward, mp_backward) : tuple[flopy.modpath.Modpath7, Modpath7]
    """
    # ── defaults from the GWF simulation ──────────────────────────────
    if mp7_ws is None:
        mp7_ws = Path(gwf.simulation.sim_path).parent / "mp7_workspace"
    mp7_ws = Path(mp7_ws).absolute()
    mp7_ws.mkdir(exist_ok=True)

    if exe_path is None:
        exe_path = "mp7"   # rely on PATH

    def _make(direction: str) -> flopy.modpath.Modpath7:
        mp = Modpath7.create_mp7(
            modelname=f"{sim_name}_mp_{direction}",
            trackdir=direction,
            flowmodel=gwf,
            model_ws=mp7_ws,
            exe_name=str(exe_path),
            rowcelldivisions=1,
            columncelldivisions=1,
            layercelldivisions=1,
        )

        # Particle locations: ensure only (k, i, j) shape
        partlocs = [(k, i, j) for (k, i, j, *_) in river_cells]

        # Create ParticleData and ParticleGroup
        particle_data = ParticleData(partlocs, structured=True, drape=0)
        pg = ParticleGroup(particledata=particle_data)

        # Assign to model's MPSIM package
        mpsim = mp.get_package("MPSIM")
        mpsim.particlegroups.clear()
        mpsim.particlegroups.append(pg)

        return mp

    mp_forward  = _make("forward")
    mp_backward = _make("backward")
    return mp_forward, mp_backward

## Simulation Settings

In [4]:
def write_models(*sims, silent=False):
    for sim in sims:
        if isinstance(sim, flopy.mf6.MFSimulation):
            sim.write_simulation(silent=silent)
        else:
            sim.write_input()

@timed
def run_models(*sims, silent=False):
    for sim in sims:
        if isinstance(sim, flopy.mf6.MFSimulation):
            print(f"Running simulation: {sim.name}")
            success, buff = sim.run_simulation(silent=silent, report=True)
        else:
            print(f"Running model: {sim.name}")
            success, buff = sim.run_model(silent=silent, report=True)
        
        if not success:
            print(f"Simulation {sim.name} failed.")
            print(buff)
            break
        else:
            print(f"Simulation {sim.name} succeeded.")

In [5]:
## Plot Groundwater Model Results
def load_head():
    # Assuming you have a head file to load
    head_file = gwf_ws / headfile
    head_obj = flopy.utils.HeadFile(head_file)
    head = head_obj.get_data()
    return head

def plot_gwf_all(gwfsim):
    # get gwf model
    gwf = gwfsim.get_model(gwf_name)
    head = load_head()

    # Load the discretization file to access model grid information
    dis = gwf.get_package("DIS")
    nlay, nrow, ncol = dis.nlay.data, dis.nrow.data, dis.ncol.data

    # Load the idomain array to identify active cells
    idomain = dis.idomain.array # No results will be visible otherwise

    # Choose the layer you want to plot, e.g., the first layer (layer 0)
    layer_to_plot = 1  # You can change this to any other layer (0-based index)

    # Extract the groundwater head for the specified layer (nrow, ncol)
    head_layer = head[layer_to_plot, :, :]

    # Mask the inactive cells in the head_layer array
    head_layer_masked = np.ma.masked_where(idomain[layer_to_plot, :, :] == 0, head_layer)

    # Plot the groundwater head for the chosen layer
    plt.figure(figsize=(10, 6))
    plt.imshow(head_layer_masked, cmap='viridis', origin='lower', extent=[0, ncol, 0, nrow])
    plt.colorbar(label='Groundwater Head (m)')
    plt.title(f'Groundwater Head at Layer {layer_to_plot + 1}')
    plt.xlabel('Column')
    plt.ylabel('Row')
    plt.show()

    # Load the surface elevation data
    surface_elevation = dis.top.array

    # Choose the layers you want to plot, e.g., the first layer (layer 0) and the last layer
    layer_to_plot_first = 0  # First layer (0-based index)
    layer_to_plot_last = nlay - 1  # Last layer (0-based index)

    # Extract the groundwater head for the specified layers (nrow, ncol)
    head_layer_first = head[layer_to_plot_first, :, :]
    head_layer_last = head[layer_to_plot_last, :, :]

    # Mask the inactive cells in the head_layer arrays
    head_layer_first_masked = np.ma.masked_where(idomain[layer_to_plot_first, :, :] == 0, head_layer_first)
    head_layer_last_masked = np.ma.masked_where(idomain[layer_to_plot_last, :, :] == 0, head_layer_last)

    # Plot the surface elevation for active cells and overlay groundwater head contours
    fig, axs = plt.subplots(1, 2, figsize=(20, 10))

    # Plot for the first layer
    top_active_first = np.ma.masked_where(idomain[layer_to_plot_first, :, :] == 0, surface_elevation)
    im1 = axs[0].imshow(top_active_first, cmap="terrain", interpolation="nearest", origin="lower",
                        extent=[0, ncol, 0, nrow], alpha=0.7)
    plt.colorbar(im1, ax=axs[0], label='Surface Elevation (m)')

    # Check if the minimum and maximum values are different before creating contour levels
    if head_layer_first_masked.min() != head_layer_first_masked.max():
        contour_first = axs[0].contour(head_layer_first_masked, levels=np.linspace(head_layer_first_masked.min(), head_layer_first_masked.max(), 10), colors='blue', extent=[0, ncol, 0, nrow])
        axs[0].clabel(contour_first, inline=True, fontsize=8, fmt='%1.1f')
    axs[0].set_title(f'Surface Elevation and Groundwater Head Contours at Layer {layer_to_plot_first + 1}')
    axs[0].set_xlabel('Column')
    axs[0].set_ylabel('Row')

    # Plot for the last layer
    top_active_last = np.ma.masked_where(idomain[layer_to_plot_last, :, :] == 0, surface_elevation)
    im2 = axs[1].imshow(top_active_last, cmap="terrain", interpolation="nearest", origin="lower",
                        extent=[0, ncol, 0, nrow], alpha=0.7)
    plt.colorbar(im2, ax=axs[1], label='Surface Elevation (m)')

    # Check if the minimum and maximum values are different before creating contour levels
    if head_layer_last_masked.min() != head_layer_last_masked.max():
        contour_last = axs[1].contour(head_layer_last_masked, levels=np.linspace(head_layer_last_masked.min(), head_layer_last_masked.max(), 10), colors='blue', extent=[0, ncol, 0, nrow])
        axs[1].clabel(contour_last, inline=True, fontsize=8, fmt='%1.1f')
    axs[1].set_title(f'Surface Elevation and Groundwater Head Contours at Layer {layer_to_plot_last + 1}')
    axs[1].set_xlabel('Column')
    axs[1].set_ylabel('Row')

    plt.tight_layout()
    plt.show()

    #---------------------- Zoom In to idomain ------------------------#
    # Choose the layers you want to plot
    layers_to_plot = [1, 19, 39]  # 1st, 20th, and 40th layers (0-based index)

    # Extract the groundwater head for the specified layers (nrow, ncol)
    head_layers = [head[layer, :, :] for layer in layers_to_plot]

    # Mask the inactive cells in the head_layer arrays
    head_layers_masked = [np.ma.masked_where(idomain[layer, :, :] == 0, head_layers[i]) for i, layer in enumerate(layers_to_plot)]

    # Determine the extent of the active cells
    active_cells = np.any(idomain, axis=0)
    active_rows, active_cols = np.where(active_cells)
    row_min, row_max = active_rows.min(), active_rows.max()
    col_min, col_max = active_cols.min(), active_cols.max()

    # Define the extent for the plots
    extent = [col_min, col_max + 1, row_min, row_max + 1]

    # Plot the surface elevation for active cells and overlay groundwater head contours
    fig, axs = plt.subplots(3, 1, figsize=(10, 30))

    for i, layer in enumerate(layers_to_plot):
        # Plot for each layer
        top_active = np.ma.masked_where(idomain[layer, :, :] == 0, surface_elevation)
        im = axs[i].imshow(top_active[row_min:row_max+1, col_min:col_max+1], cmap="terrain", interpolation="nearest", origin="lower",
                           extent=extent, alpha=0.7)
        plt.colorbar(im, ax=axs[i], label='Surface Elevation (m)')
        contour = axs[i].contour(head_layers_masked[i][row_min:row_max+1, col_min:col_max+1], levels=np.linspace(head_layers_masked[i].min(), head_layers_masked[i].max(), 10), colors='blue', extent=extent)
        axs[i].clabel(contour, inline=True, fontsize=8, fmt='%1.1f')
        axs[i].set_title(f'Surface Elevation and Groundwater Head Contours at Layer {layer + 1}')
        axs[i].set_xlabel('Column')
        axs[i].set_ylabel('Row')

    plt.tight_layout()
    plt.show()

    #---------------------- 3D Plot of the Model ------------------------#
    top = dis.top.array
    botm = dis.botm.array
    idomain = dis.idomain.array  # Assuming idomain is part of the dis object

    # Combine top and botm to get the elevation data for all layers
    elevation_data = np.concatenate(([top], botm), axis=0)

    # Get the number of rows and columns
    nrows, ncols = top.shape

    # Layer to plot for terrain
    terrain_layer = 0

    # Create a meshgrid for x and y coordinates
    x = np.linspace(0, ncols - 1, ncols)
    y = np.linspace(0, nrows - 1, nrows)
    x, y = np.meshgrid(x, y)

    # Mask the elevation data using the idomain array
    #z = np.ma.masked_where(idomain[terrain_layer, :, :] == 0, elevation_data[terrain_layer, :, :])

    # Set up plot
    fig, ax = plt.subplots(subplot_kw=dict(projection='3d'))

    # Light source for hillshading
    ls = LightSource(270, 45)

    # Plot the masked elevation data
    z = elevation_data[terrain_layer, :, :]
    rgb = ls.shade(z, cmap=cm.gist_earth, vert_exag=0.1, blend_mode='soft')
    surf = ax.plot_surface(x, y, z, rstride=1, cstride=1, facecolors=rgb,
                        linewidth=0, antialiased=False, shade=False)

    # Set plot labels and title
    ax.set_title('3D Terrain Elevation')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ax.set_zlabel('Elevation (ft)')

    plt.show()

## Code from:
# https://github.com/matplotlib/matplotlib/tree/cfe5bf75eaf378b9523830908036f2123acfe4e7/examples/frontpage/3D.py

## Plot Particle Tracking Results

In [6]:
def plot_modpath7_results(mpnamf, ws, gwf):
    """
    Process MODPATH 7 results to filter hyporheic flow paths and calculate:
    A) Flow path length distribution
    B) Residence time distribution
    C) Spatial extent of the hyporheic zone

    Parameters:
    - mpnamf: str, MODPATH 7 model name
    - ws: str, workspace directory for MODPATH 7
    - gwf: flopy.mf6.ModflowGwf, MODFLOW 6 groundwater flow model
    """
    # Load forward tracking pathline data
    pathline_file_forward = os.path.join(ws, f"{mpnamf_forward}.mppth")
    pathlines_forward = flopy.utils.PathlineFile(pathline_file_forward).get_alldata()

    # Load backward tracking pathline data
    pathline_file_backward = os.path.join(ws, f"{mpnamf_backward}.mppth")
    pathlines_backward = flopy.utils.PathlineFile(pathline_file_backward).get_alldata()

    # Combine forward and backward pathlines
    all_pathlines = list(pathlines_forward) + list(pathlines_backward)

    # Filter hyporheic flow paths (start and end at the riverbed)
    hyporheic_paths = []
    for pathline in all_pathlines:
        start_layer, start_row, start_column = pathline["k0"], pathline["i0"], pathline["j0"]
        end_layer, end_row, end_column = pathline["k"], pathline["i"], pathline["j"]

        # Check if both start and end points are in riverbed cells
        if is_riverbed_cell(start_layer, start_row, start_column, gwf) and \
           is_riverbed_cell(end_layer, end_row, end_column, gwf):
            hyporheic_paths.append(pathline)

    # Calculate flow path lengths
    flow_path_lengths = [pathline.get_length() for pathline in hyporheic_paths]

    # Calculate residence times
    residence_times = [pathline["time"] for pathline in hyporheic_paths]

    # Calculate spatial extent of the hyporheic zone
    hyporheic_extent = calculate_hyporheic_extent(hyporheic_paths, gwf)

    # Plot flow path length distribution
    plt.figure(figsize=(10, 6))
    plt.hist(flow_path_lengths, bins=20, color="skyblue", edgecolor="black")
    plt.title("Flow Path Length Distribution")
    plt.xlabel("Flow Path Length")
    plt.ylabel("Frequency")
    plt.grid()
    plt.show()

    # Plot residence time distribution
    plt.figure(figsize=(10, 6))
    plt.hist(residence_times, bins=20, color="lightgreen", edgecolor="black")
    plt.title("Residence Time Distribution")
    plt.xlabel("Residence Time")
    plt.ylabel("Frequency")
    plt.grid()
    plt.show()

    # Print spatial extent of the hyporheic zone
    print(f"Spatial Extent of Hyporheic Zone:")
    print(f"  Total Volume: {hyporheic_extent['volume']:.2f} cubic units")
    print(f"  Width: {hyporheic_extent['width']:.2f} units")
    print(f"  Depth: {hyporheic_extent['depth']:.2f} units")

def is_riverbed_cell(layer, row, column, gwf):
    """
    Check if a cell is part of the riverbed.

    Parameters:
    - layer: int, layer index
    - row: int, row index
    - column: int, column index
    - gwf: flopy.mf6.ModflowGwf, MODFLOW 6 groundwater flow model

    Returns:
    - bool: True if the cell is part of the riverbed, False otherwise
    """
    # Example logic: Check if the cell is in the riverbed layer and has a river stage
    river_stage = gwf.riv.stress_period_data.array["stage"]
    bed_elevation = gwf.dis.top.array[row, column] - gwf.dis.botm.array[layer, row, column]
    return river_stage[row, column] > bed_elevation


def calculate_hyporheic_extent(hyporheic_paths, gwf):
    """
    Calculate the spatial extent of the hyporheic zone.

    Parameters:
    - hyporheic_paths: list, filtered hyporheic flow paths
    - gwf: flopy.mf6.ModflowGwf, MODFLOW 6 groundwater flow model

    Returns:
    - dict: Dictionary containing total volume, width, and depth of the hyporheic zone
    """
    # Extract unique cells from hyporheic paths
    unique_cells = set((pathline["k"], pathline["i"], pathline["j"]) for pathline in hyporheic_paths)

    # Calculate spatial extent
    cell_volume = gwf.dis.delr.array[0] * gwf.dis.delc.array[0] * gwf.dis.thickness.array[0]
    total_volume = len(unique_cells) * cell_volume
    width = gwf.dis.delc.array[0] * len(set(cell[2] for cell in unique_cells))  # Unique columns
    depth = gwf.dis.thickness.array[0] * len(set(cell[0] for cell in unique_cells))  # Unique layers

    return {"volume": total_volume, "width": width, "depth": depth}

## Run Simulations

In [ ]:
#---------------------- Simulation Scenario ------------------------#
def scenario(
    river_cells,
    cfg,
    chd_data,
    idomain,
    mp7_ws,
    mp7_exe_path,
    write=True,
    run=True,
    plot=True,
    silent=False,
):
    """
    Orchestrates the building, running, and plotting of the MODFLOW 6 and MODPATH 7 models.
    """
    # Build the GWF model using the new signature
    gwfsim, gwf = build_gwf_model(cfg, chd_data, idomain)

    print("GWF model built:", gwfsim)

    if write:
        write_models(gwfsim, silent=silent)
        print("GWF files written to:", cfg.gwf_ws)

    if run:
        print("Running MODFLOW model...")
        run_models(gwfsim, silent=False)
        print("FINISHED! Running GWF MODFLOW 6")

    if plot:
        print("Plotting Groundwater Flow Model")
        plot_gwf_all(gwfsim)

    try:
        # Build both MODPATH 7 forward and backward models
        mp_forward, mp_backward = build_particle_models(
            cfg.sim_name, gwf, river_cells, mp7_ws=mp7_ws, exe_path=mp7_exe_path
        )

        print("MODPATH 7 forward model built:", mp_forward)
        print("MODPATH 7 backward model built:", mp_backward)

        if write:
            write_models(mp_forward, silent=silent)
            write_models(mp_backward, silent=silent)
            print("MODPATH 7 files written to:", mp7_ws)

        if run:
            print("Running MODPATH 7 forward model...")
            run_models(mp_forward, silent=silent)
            print("FINISHED! Running MODPATH 7 forward model")

            print("Running MODPATH 7 backward model...")
            run_models(mp_backward, silent=silent)
            print("FINISHED! Running MODPATH 7 backward model")

        if plot:
            print("Plotting MODPATH 7 Results")
            plot_modpath7_results(
                #mpnamf_forward=f"{cfg.sim_name}_mp_forward",
                #mpnamf_backward=f"{cfg.sim_name}_mp_backward",
                #ws=str(mp7_ws),
                #gwf=gwf
            )

    except Exception as e:
        print(f"An error occurred while building, running, or plotting the MODPATH 7 models: {e}")

# Example usage (assuming you have constructed cfg, chd_data, idomain, river_cells):
scenario(
    river_cells=river_cells,
    cfg=cfg,
    chd_data=chd_data,
    idomain=idomain,
    mp7_ws="mp7_workspace",
    mp7_exe_path="mp7",
    write=True,
    run=True,
    plot=True,
    silent=True
)

Building GWF model for Hyporheic_Project
✅ Assigned 14493 unique CHD boundary cells.


GWF model built: sim_name = gwf_model
sim_path = C:\Users\u4eeevmq\Documents\Python\HyporheicFloPy\HP_workspace\gwf_workspace
exe_name = C:\Users\u4eeevmq\Documents\Python\Flo_Py\flopy\modflowExe\mf6.exe

###################
Package mfsim.nam
###################

package_name = mfsim.nam
filename = mfsim.nam
package_type = nam
model_or_simulation_package = simulation
simulation_name = gwf_model


###################
Package tdis
###################

package_name = tdis
filename = gwf_model.tdis
package_type = tdis
model_or_simulation_package = simulation
simulation_name = gwf_model


###################
Package ims_-1
###################

package_name = ims_-1
filename = gwf_model.ims
package_type = ims
model_or_simulation_package = simulation
simulation_name = gwf_model


@@@@@@@@@@@@@@@@@@@@
Model gwf_model
@@@@@@@@@@@@@@@@@@@@

name = gwf_model
model_type = gwf6
version = mf6
model_relative_path = .

###################
Package dis
###################

package_name = dis
filename = 

GWF files written to: HP_workspace\gwf_workspace
Running MODFLOW model...
Running simulation: gwf_model
